# Task 4
This serves as a template which will guide you through the implementation of this task. It is advised to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [1]:
import os
# We disable low-level log outputs by default to keep the terminal clean
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import wandb

# Add any other imports you need here

from transformers import AutoTokenizer, AutoModelForSequenceClassification # for transformers

c:\Users\carlo\anaconda3\envs\IML\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Depending on your approach, you might need to adapt the structure of this template or parts not marked by TODOs.
It is not necessary to completely follow this template. Feel free to add more code and delete any parts that are not required.

In [2]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32  # standard for DistillBERT # TODO: Set the batch size according to both training performance and available memory
NUM_EPOCHS = 5  # just to start with TODO: Set the number of epochs

train_val = pd.read_csv("train.csv")
test_val = pd.read_csv("test_no_score.csv")

train_val["text"] = train_val["title"].fillna("") + " - " + train_val["sentence"].fillna("")
train_val["label"] = train_val["score"] # Rename to match template

test_val["text"] = test_val["title"].fillna("") + " - " + test_val["sentence"].fillna("")

In [3]:
train_val.head(10)

,title,sentence,score,text,label
0,A strange album,This one is a bit strange. On one hand I think...,1,A strange album - This one is a bit strange. O...,1
1,Tales of King Arthur,The books were received in a timely manner. I ...,1,Tales of King Arthur - The books were received...,1
2,Great Tribute to the Boykin Spaniel,This is a great book to share with your childr...,1,Great Tribute to the Boykin Spaniel - This is ...,1
3,A Bit of a Disappointment,I think Phil's ego filled up most of the pages...,0,A Bit of a Disappointment - I think Phil's ego...,0
4,Compare I know what I saw/ with Out of the blu...,I recently purchased these two dvd's. It was a...,0,Compare I know what I saw/ with Out of the blu...,0
5,Stevie and his band are incredible!,"I ordered this DVD with a bunch of other ""Funk...",1,Stevie and his band are incredible! - I ordere...,1
6,Silence of the North,I was thrilled to find this movie I was lookin...,1,Silence of the North - I was thrilled to find ...,1
7,"The cover and paper quality, and graph color a...",The book's content is rich and meaningful. But...,0,"The cover and paper quality, and graph color a...",0
8,Iron Man 2 Review,Not as good as the first Iron Man. Iron Man 2 ...,0,Iron Man 2 Review - Not as good as the first I...,0
9,Cream - 'I Feel Free: Ultimate Cream' (Polydor),A superb 23 track collection of Cream's repert...,1,Cream - 'I Feel Free: Ultimate Cream' (Polydor...,1


In [4]:
test_val.head()

,title,sentence,text
0,Give it time,"If you are looking for another Murmer, look so...",Give it time - If you are looking for another ...
1,"Useless, with the exception of checking your e...","As others have stated, this device is USB 1.1,...","Useless, with the exception of checking your e..."
2,The Album That Started It All For Martina...,Still one of my favorite Martina albums.This i...,The Album That Started It All For Martina... -...
3,Not up to some other versions,This production of the classic was just okay. ...,Not up to some other versions - This productio...
4,Great CD,This cd has a good selection of Prince great h...,Great CD - This cd has a good selection of Pri...


In [5]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels
        # DistilBERT is a lighter, faster version of BERT perfectly suited for this
        self.tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
        self.max_length = 128 # Cuts off extremely long reviews to save memory

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts[index])
        
        # The tokenizer handles padding, cutting, and converting words to numbers
        inputs = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        # squeeze(0) removes the batch dimension the tokenizer adds by default
        item = {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0)
        }
        
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
            
        return item

In [6]:
train_dataset = SentimentDataset(train_val["text"].tolist(), train_val["label"].tolist())
test_dataset = SentimentDataset(test_val["text"].tolist())

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True,
                          num_workers=0,         # If you want to utilize multi-processing, set this to the number of your available cores!
                          pin_memory=True)
test_loader = DataLoader(dataset=test_dataset,
                         batch_size=BATCH_SIZE,
                         shuffle=False,
                         num_workers=0,          # If you want to utilize multi-processing, set this to the number of your available cores!
                         pin_memory=True)
# Additional code if needed

In [7]:
# TODO: Fill out SentimentClassifier
    
class SentimentClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # Load the pre-trained model and tell it we want 2 output classes (Positive/Negative)
        self.model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
        
        # --- FREEZE THE BASE WEIGHTS (To pass the baseline quickly!) ---
        for param in self.model.distilbert.parameters():
            param.requires_grad = False
            
    def forward(self, input_ids, attention_mask):
        # The model returns a special dictionary-like object, we want the raw predictions (logits)
        output = self.model(input_ids=input_ids, attention_mask=attention_mask)
        return output.logits


model = SentimentClassifier().to(DEVICE)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2497.46it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
# TODO: Setup loss function, optimizer, and scheduler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

wandb.init(
    entity="dariofranzen-team",  # Your specific team name
    project="iml-task4",         # Your specific project name
    name="DistilBERT-Baseline_Corrected",  # A recognizable name for this run
    config={
        "learning_rate": 2e-3,
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "architecture": "DistilBERT (Frozen)"
    }
)

model.train()
# Training Loop
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    
    for batch in tqdm(train_loader, total=len(train_loader)):
        # Move inputs to GPU
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        
        # Forward pass
        logits = model(input_ids, attention_mask)
        
        # Calculate loss and step
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)

    print(f"Epoch {epoch} | Training Loss: {epoch_loss:.4f}")

    wandb.log({
        "epoch": epoch,
        "train_loss": epoch_loss,
        "learning_rate": optimizer.param_groups[0]['lr']
    })
        
    print(f"Epoch {epoch} | Training Loss: {running_loss / len(train_loader):.4f}")
    scheduler.step(running_loss)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\carlo\_netrc.
wandb: Currently logged in as: carlo-rossetto (dariofranzen-team) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 391/391 [03:31<00:00,  1.85it/s]


Epoch 0 | Training Loss: 0.4316
Epoch 0 | Training Loss: 0.4316


100%|██████████| 391/391 [03:31<00:00,  1.85it/s]


Epoch 1 | Training Loss: 0.3875
Epoch 1 | Training Loss: 0.3875


...

Epoch 2 | Training Loss: 0.3769
Epoch 2 | Training Loss: 0.3769


100%|██████████| 391/391 [03:33<00:00,  1.83it/s]


Epoch 3 | Training Loss: 0.3756
Epoch 3 | Training Loss: 0.3756


100%|██████████| 391/391 [03:31<00:00,  1.85it/s]

Epoch 4 | Training Loss: 0.3606
Epoch 4 | Training Loss: 0.3606


In [9]:
model.eval()
with torch.no_grad():
    results = []
    for batch in tqdm(test_loader, total=len(test_loader)):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        # TODO: Set up evaluation loop

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        # Get the raw scores (logits)
        logits = model(input_ids, attention_mask)
        
        # The prediction is the index (0 or 1) with the highest score
        predictions = torch.argmax(logits, dim=1)
        
        # Move back to CPU and add to our list
        results.append(predictions.cpu().numpy())

    with open("result.txt", "w") as f:
        for val in np.concatenate(results):
            f.write(f"{val}\n")

wandb.finish()

100%|██████████| 32/32 [00:16<00:00,  1.93it/s]


epoch,▁▃▅▆█
learning_rate,▁▁▁▁▁
train_loss,█▄▃▂▁
epoch,4
learning_rate,0.002
train_loss,0.36057
